# 04h — Optuna sur le LSTM et le CNN 1D, FD001 et FD002

**Question.** Le LSTM et le CNN 1D du notebook 03b avaient des réglages fixés à l'avance (2 × 64 unités, fenêtre de 30 cycles, pas de 1e-3…), jamais optimisés. Avec la même méthode que pour les arbres (Optuna au 04d et au 04f, confirmation sur des découpages neufs), peut-on faire mieux, et sur chaque sous-jeu séparément ?

**Réponse courte.**

- **FD001** : oui, nettement. En validation (3 découpages par moteur jamais utilisés pour choisir), le LSTM passe de 9,34 à 7,85 et le CNN 1D de 9,60 à 8,30. Le modèle retenu est la moyenne de **5 LSTM** (5 graines) : test officiel **11,16** contre la RUL brute (10,09 contre la RUL plafonnée à 125), contre 12,29 pour le LSTM du 03b et 12,73 pour le XGBoost final.
- **FD002** : oui pour le LSTM (10,28 → 9,90 en validation), non pour le CNN 1D (gain non confirmé, il garde les réglages du 03b). Modèle retenu : 5 LSTM, test 22,38 brut, 10,62 plafonné.
- **FD003 et FD004** : pas de recherche propre ; ils gardent les réglages de FD001 du 03b, et on le dit.

**Ce notebook ne relance aucun calcul.** Il relit les résultats du calcul de nuit du 2026-11-07 (`bash scripts/nuit_04h.sh`, environ 12 h sur tout le processeur), copiés et versionnés dans `docs/resultats/04h/`. Le code qui les a produits : `src/optimisation_reseaux.py` et `scripts/optimiser_reseaux.py`. Les réglages retenus sont recopiés dans `src/optimisation_reseaux.py` (`CONFIGS_RESEAUX_RETENUES`), pour réentraîner sans relancer Optuna.

Plan : 1. le protocole ; 2. la recherche Optuna ; 3. quels réglages comptent ; 4. la confirmation sur des découpages neufs ; 5. l'ablation des entrées ; 6. les ensembles ; 7. le test officiel ; 8. l'évolution, étape par étape ; 9. la littérature ; 10. où sont les réglages ; 11. les limites.

In [ ]:
import json
import os
import sys

sys.path.insert(0, "..")  # pour importer src/ depuis notebooks/

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd

# src.config_modele charge xgboost avant torch (ordre voulu, voir src/deep_learning.py)
from src.config_modele import CONFIG_XGBOOST_FINAL
from src import optimisation_reseaux as opt

DOSSIER = os.path.join("..", "docs", "resultats", "04h")  # copie versionnee des resultats de la nuit
PREFIXES = {"FD001": "04h_", "FD002": "04h_fd002_"}  # debut des noms de fichiers de chaque sous-jeu
RESEAUX = {"lstm": "LSTM", "cnn1d": "CNN 1D"}  # nom de code -> nom affiche
DECOUPAGES_NEUFS = ["1000", "1001", "1002"]  # graines des 3 decoupages jamais utilises pour choisir
COULEURS = {"lstm": "#0072B2", "cnn1d": "#E69F00", "reference": "#999999"}

optuna.logging.set_verbosity(optuna.logging.WARNING)


def lire_json(sous_jeu, nom):
    """Relit un fichier de resultats d'un sous-jeu (ex. lire_json("FD001", "confirmation_lstm"))."""
    chemin = os.path.join(DOSSIER, PREFIXES[sous_jeu] + nom + ".json")
    with open(chemin, encoding="utf-8") as fichier:
        return json.load(fichier)


def moyenne_neufs(resume_cv):
    """RMSE de validation moyen sur les 3 decoupages neufs (resume_cv : graine -> {"rmse_cv_moyen", ...})."""
    valeurs = []
    for graine in DECOUPAGES_NEUFS:
        valeurs.append(resume_cv[graine]["rmse_cv_moyen"])
    return float(np.mean(valeurs))


print("fichiers relus depuis", os.path.abspath(DOSSIER), ":", len(os.listdir(DOSSIER)), "fichiers")

## 1. Le protocole, écrit avant les calculs

Les règles sont écrites dans le code (`scripts/optimiser_reseaux.py`, `TEXTE_REGLE_*`) **avant** de lancer la recherche, puis recopiées dans chaque fichier de résultats. On les relit ici telles qu'elles ont servi. L'idée, en trois temps :

1. **Chercher** : Optuna essaie des réglages et note chacun par le RMSE moyen de validation croisée (5 plis par moteur, graine 42 : les plis des arbres et du 03b). Un essai clairement mauvais après un pli est abandonné (« élagué ») pour gagner du temps.
2. **Confirmer** : le meilleur essai a été choisi sur ces mêmes plis, donc son score est **optimiste** (il a « gagné » parmi plus de 100 essais, en partie par chance). On le compare donc à la référence du 03b sur **3 découpages neufs**, avec un seuil fixé à l'avance.
3. **Tester une seule fois** : le jeu de test officiel n'intervient dans aucun choix ; il est lu une fois, à la fin, pour les modèles retenus.

In [ ]:
# Les regles, telles qu'elles sont enregistrees dans les fichiers de resultats
print("RECHERCHE :", lire_json("FD001", "optuna_lstm")["regle"], "\n")
print("CONFIRMATION :", lire_json("FD001", "confirmation_lstm")["regle"], "\n")
print("ENSEMBLES :", lire_json("FD001", "ensemble")["regle"], "\n")

# FD002 : la regle du changement (chaque sous-jeu a sa propre recherche), enregistree dans l'etude Optuna
stockage = "sqlite:///" + os.path.abspath(os.path.join(DOSSIER, "04h_optuna.db")).replace("\\", "/")
etude_fd002 = optuna.load_study(study_name="04h_fd002_lstm", storage=stockage)
print("SOUS-JEUX :", etude_fd002.user_attrs["regle_sous_jeux"])

## 2. La recherche Optuna

Quatre études : LSTM et CNN 1D, sur FD001 et sur FD002. Espace de recherche (`src/optimisation_reseaux.py`, `config_depuis_essai`) :

- LSTM : taille cachée (32 à 192 unités), 1 à 3 couches, dropout de 0 à 0,5 ;
- CNN 1D : 2 à 4 blocs, 16 à 128 filtres, noyau de 3 à 9 cycles, dropout ;
- pour les deux : pas d'Adam (3e-4 à 5e-3), taille des lots (128, 256, 512), longueur de fenêtre (20 à 50 cycles), et k de l'écart à l'état initial (20, 25 ou 30 sur FD001 ; 20 seulement sur FD002, dont le plus court moteur test n'a que 21 cycles).

Sur FD001, l'essai n° 0 du LSTM est la référence du 03b : on doit y retrouver son RMSE de validation (8,94), ce qui contrôle que le protocole est bien le même.

In [ ]:
# Une ligne par etude : nombre d'essais, meilleur RMSE de validation (plis 42), essai qui l'a donne
lignes = []
for sous_jeu in PREFIXES:
    for reseau in RESEAUX:
        resume = lire_json(sous_jeu, "optuna_" + reseau)
        lignes.append({
            "sous-jeu": sous_jeu, "reseau": RESEAUX[reseau],
            "essais lances": resume["nb_essais"], "complets": resume["nb_complets"], "elagues": resume["nb_elagues"],
            "meilleur essai": resume["meilleur_numero"], "meilleur RMSE (plis 42)": round(resume["meilleur_rmse_cv"], 3),
        })
tableau_etudes = pd.DataFrame(lignes)

# controle : l'essai n° 0 du LSTM FD001 est la reference du 03b
essais_lstm_fd001 = pd.read_csv(os.path.join(DOSSIER, "04h_optuna_lstm.csv"))
print("essai n° 0 du LSTM FD001 (reference du 03b) :", round(float(essais_lstm_fd001.loc[0, "value"]), 4), "(03b : 8,9421)")
tableau_etudes

Les essais « lancés » comprennent aussi des essais interrompus (`FAIL`) : la première nuit, la machine s'est mise en veille, et plusieurs arrêts du calcul en journée ont laissé des essais inachevés. Ils sont exclus de tout : seuls les essais complets ou élagués comptent.

In [ ]:
# --- Figure : historique de chaque recherche ---
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, sous_jeu in enumerate(PREFIXES):
    for j, reseau in enumerate(RESEAUX):
        ax = axes[i, j]
        essais = pd.read_csv(os.path.join(DOSSIER, PREFIXES[sous_jeu] + "optuna_" + reseau + ".csv"))
        complets = essais[essais["state"] == "COMPLETE"].sort_values("number")
        ax.scatter(complets["number"], complets["value"], s=14, color=COULEURS[reseau], alpha=0.6, label="essai complet")
        meilleur_jusque_la = complets["value"].cummin()  # meilleur RMSE vu jusqu'a cet essai
        ax.step(complets["number"], meilleur_jusque_la, where="post", color="black", label="meilleur jusque-la")
        confirmation = lire_json(sous_jeu, "confirmation_" + reseau)
        reference_42 = confirmation["cv_reference"]["42"]["rmse_cv_moyen"]  # reference du 03b, memes plis
        ax.axhline(reference_42, color=COULEURS["reference"], linestyle="--", label=f"reference 03b ({reference_42:.2f})")
        ax.set_ylim(meilleur_jusque_la.min() - 0.5, reference_42 + 3)
        ax.set_title(f"{sous_jeu}, {RESEAUX[reseau]}")
        ax.set_xlabel("numero de l'essai")
        ax.set_ylabel("RMSE de validation (plis 42)")
        ax.legend(fontsize=8)
fig.suptitle("Optuna trouve-t-il mieux que les reglages fixes du 03b ?")
plt.tight_layout()
plt.show()

**Lecture.** Dans les quatre études, Optuna descend nettement sous la référence du 03b (ligne pointillée). Pour trois d'entre elles, le meilleur score ne s'améliore plus pendant de nombreux essais (dernier record : essai 50 sur 132 pour le LSTM de FD001, 74 sur 110 pour le LSTM de FD002, 47 sur 81 pour le CNN de FD002) : des essais en plus n'apporteraient probablement pas grand-chose. **Le CNN de FD001 fait exception** : son dernier record est à l'essai 77 sur 89, la recherche n'était peut-être pas finie (voir les limites). Sur FD002, beaucoup d'essais sont élagués (71 sur 111 pour le LSTM) : l'élagage a abandonné tôt les essais sans espoir, ce qui a rendu la recherche faisable en une nuit. Ces scores sont mesurés sur les plis mêmes qui ont servi à choisir : ils sont optimistes, d'où la section 4.

## 3. Quels réglages comptent le plus ?

Optuna estime l'importance de chaque réglage (méthode fANOVA : quelle part des écarts de RMSE entre essais s'explique par ce réglage). C'est une indication, pas une preuve : elle dépend des essais qu'Optuna a choisi de faire. Un réglage qui n'a qu'une seule valeur possible (k sur FD002) est exclu.

In [ ]:
# --- Figure : importance de chaque reglage, pour chaque etude ---
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, sous_jeu in enumerate(PREFIXES):
    for j, reseau in enumerate(RESEAUX):
        etude = optuna.load_study(study_name=PREFIXES[sous_jeu] + reseau, storage=stockage)
        complets = etude.trials_dataframe(attrs=("params", "state"))
        complets = complets[complets["state"] == "COMPLETE"]
        reglages = []  # seulement les reglages qui ont pris plusieurs valeurs
        for colonne in complets.columns:
            if colonne.startswith("params_") and complets[colonne].nunique() > 1:
                reglages.append(colonne.replace("params_", ""))
        evaluateur = optuna.importance.FanovaImportanceEvaluator(seed=42)  # graine fixee : resultat reproductible
        importances = optuna.importance.get_param_importances(etude, evaluator=evaluateur, params=reglages)
        noms = list(importances.keys())[::-1]  # le plus important en haut
        valeurs = list(importances.values())[::-1]
        axes[i, j].barh(noms, valeurs, color=COULEURS[reseau])
        axes[i, j].set_title(f"{sous_jeu}, {RESEAUX[reseau]}")
        axes[i, j].set_xlabel("importance (part des ecarts de RMSE)")
fig.suptitle("Quels reglages comptent le plus dans le RMSE ?")
plt.tight_layout()
plt.show()

In [ ]:
# Les 5 meilleurs essais de chaque etude : les tendances communes se lisent ici
for sous_jeu in PREFIXES:
    for reseau in RESEAUX:
        essais = pd.read_csv(os.path.join(DOSSIER, PREFIXES[sous_jeu] + "optuna_" + reseau + ".csv"))
        complets = essais[essais["state"] == "COMPLETE"].sort_values("value").head(5)
        colonnes = ["number", "value"]
        for colonne in complets.columns:
            if colonne.startswith("params_"):
                colonnes.append(colonne)
        print(f"--- {sous_jeu}, {RESEAUX[reseau]} : 5 meilleurs essais")
        print(complets[colonnes].round(4).to_string(index=False), "\n")

**Lecture.**

- **Ce qui compte le plus** : la **longueur de fenêtre** pour le CNN (77 % de l'importance sur FD001, 63 % sur FD002) ; pour le LSTM de FD001, k de l'écart à l'état initial (34 %), puis la longueur (19 %) et le pas d'Adam (17 %) ; pour le LSTM de FD002, le **pas d'Adam** (59 %).
- **Tendances communes aux meilleurs essais** : **fenêtres courtes** (20 cycles au lieu de 30, dans presque tous les 5 meilleurs essais des quatre études) et **réseaux petits** (32 unités ou 32 filtres dans la grande majorité). Pour le LSTM : lots de 128 fenêtres et pas d'Adam de 2 à 5e-3, plus grand que la valeur par défaut (1e-3) ; sur FD001, k = 30 dans les 5 meilleurs essais ; sur FD002, 3 couches dans les 5 meilleurs. Pour le CNN, les lots de 256 dominent.
- Autrement dit, le 03b avait des réseaux plus gros, et des fenêtres plus longues, que nécessaire pour 100 à 260 moteurs d'entraînement.

## 4. Le gain tient-il sur des découpages jamais vus ?

Chaque meilleur essai est comparé à la référence du 03b (même réseau, réglages du 03b) sur 3 découpages par moteur **neufs** (graines 1000, 1001, 1002). Il est retenu seulement si les trois critères écrits avant le calcul sont remplis : (a) gain moyen d'au moins 0,3 cycle, (b) meilleur sur les 3 découpages, (c) test de Wilcoxon apparié sur le RMSE par moteur, p < 0,05.

In [ ]:
lignes = []
for sous_jeu in PREFIXES:
    for reseau in RESEAUX:
        confirmation = lire_json(sous_jeu, "confirmation_" + reseau)
        comparaison = confirmation["comparaison"]
        gains = []
        for gain in comparaison["gains_par_decoupage"]:
            gains.append(round(gain, 2))
        lignes.append({
            "sous-jeu": sous_jeu, "reseau": RESEAUX[reseau], "essai Optuna": confirmation["essai_optuna"],
            "plis 42 (choix)": round(confirmation["cv_candidat"]["42"]["rmse_cv_moyen"], 3),
            "reference 03b (neufs)": round(moyenne_neufs(confirmation["cv_reference"]), 3),
            "candidat (neufs)": round(moyenne_neufs(confirmation["cv_candidat"]), 3),
            "gains par decoupage": gains, "gain moyen": round(comparaison["gain_moyen"], 3),
            "p Wilcoxon": f"{comparaison['p_wilcoxon']:.1e}",
            "a": comparaison["critere_a"], "b": comparaison["critere_b"], "c": comparaison["critere_c"],
            "RETENU": comparaison["retenu"],
        })
tableau_confirmation = pd.DataFrame(lignes)
tableau_confirmation

In [ ]:
# --- Figure : RMSE de validation sur chaque decoupage neuf, reference contre candidat ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
numero = 0
for sous_jeu in PREFIXES:
    for reseau in RESEAUX:
        confirmation = lire_json(sous_jeu, "confirmation_" + reseau)
        reference = []
        candidat = []
        for graine in DECOUPAGES_NEUFS:
            reference.append(confirmation["cv_reference"][graine]["rmse_cv_moyen"])
            candidat.append(confirmation["cv_candidat"][graine]["rmse_cv_moyen"])
        positions = np.arange(len(DECOUPAGES_NEUFS))
        ax = axes[numero]
        ax.bar(positions - 0.2, reference, width=0.4, color=COULEURS["reference"], label="reference 03b")
        ax.bar(positions + 0.2, candidat, width=0.4, color=COULEURS[reseau], label="meilleur essai Optuna")
        ax.set_xticks(positions)
        ax.set_xticklabels(DECOUPAGES_NEUFS)
        ax.set_xlabel("graine du decoupage")
        ax.set_ylim(min(candidat) - 1.5, max(reference) + 0.5)
        statut = "retenu" if confirmation["comparaison"]["retenu"] else "NON retenu"
        ax.set_title(f"{sous_jeu}, {RESEAUX[reseau]} : {statut}")
        numero = numero + 1
axes[0].set_ylabel("RMSE de validation (cycles)")
axes[0].legend(fontsize=8)
fig.suptitle("Le gain d'Optuna tient-il sur des decoupages jamais utilises pour choisir ?")
plt.tight_layout()
plt.show()

**Lecture.**

- Le score des plis 42 (celui qui a servi à choisir) est plus bas que celui des découpages neufs : sur FD001, 7,48 au moment du choix contre 7,85 ensuite pour le LSTM. C'est l'**optimisme** attendu d'un « meilleur parmi 100 essais », et c'est pourquoi on ne garde que les chiffres des découpages neufs.
- **FD001** : les deux réseaux passent largement (gains de 1,3 à 1,5 cycle, meilleurs sur les 3 découpages).
- **FD002, LSTM** : gain plus petit (0,38) mais régulier (0,32 à 0,50) et significatif (p = 0,002) : retenu.
- **FD002, CNN 1D** : même gain moyen (0,38), meilleur sur les 3 découpages mais faiblement sur le troisième (0,19), et surtout le test de Wilcoxon par moteur donne p = 0,25 (critère c non rempli) : **non retenu**, le CNN de FD002 garde les réglages du 03b. La règle ne regarde pas seulement la moyenne : elle demande que le gain soit régulier d'un moteur à l'autre, et pas dû au hasard.

## 5. Ablation des entrées (méthode du 04e)

Sur le LSTM retenu, on change **une famille d'entrées à la fois** et on applique la même règle, sur les mêmes 3 découpages neufs :

- `pente_30`, `pente_50` : on ajoute la pente glissante de chaque capteur sur 30 ou 50 cycles (famille F2, deuxième meilleure famille au 04e) ;
- `sans_capteurs` : on retire les capteurs lissés, il ne reste que les écarts à l'état initial et `time_cycles` (version plus simple : à garder si elle fait aussi bien).

In [ ]:
lignes = []
for sous_jeu in PREFIXES:
    ablation = lire_json(sous_jeu, "ablation_lstm")
    base = moyenne_neufs(ablation["cv_base"])
    for nom in ablation["variantes"]:
        variante = ablation["variantes"][nom]
        lignes.append({
            "sous-jeu": sous_jeu, "variante": nom, "LSTM retenu": round(base, 3),
            "variante (neufs)": round(moyenne_neufs(variante["cv"]), 3),
            "gain": round(variante["comparaison"]["gain_moyen"], 3),
            "p Wilcoxon": f"{variante['comparaison']['p_wilcoxon']:.1e}",
            "RETENU": variante["comparaison"]["retenu"],
        })
pd.DataFrame(lignes)

**Lecture.** Aucune variante n'est retenue. Ajouter la pente dégrade FD001 (−0,4 cycle) et n'apporte presque rien sur FD002 (+0,03 et +0,14, sous le seuil de 0,3). Retirer les capteurs dégrade nettement les deux sous-jeux (−1,6 et −1,3) : les capteurs lissés apportent une information que l'écart à l'état initial seul ne donne pas. Les entrées du 03b (capteurs lissés + écarts à l'état initial + `time_cycles`) restent donc les bonnes ; seule la valeur de k change (30 sur FD001).

## 6. Ensembles : faut-il moyenner plusieurs réseaux ?

Un réseau dépend de sa graine (poids initiaux, ordre des lots). Moyenner les prédictions de plusieurs réseaux entraînés avec des graines différentes réduit cette part de hasard. Deux ensembles, moyenne simple à poids égaux (rien n'est ajusté), comparés au LSTM retenu seul (graine 42) sur les mêmes 3 découpages neufs :

- `lstm_5_graines` : 5 LSTM, graines 42 à 46 ;
- `lstm_cnn_10` : ces 5 LSTM et 5 CNN 1D.

In [ ]:
lignes = []
for sous_jeu in PREFIXES:
    ensembles = lire_json(sous_jeu, "ensemble")
    seul = moyenne_neufs(ensembles["membres"]["lstm_g42"])
    for nom in ensembles["ensembles"]:
        resultat = ensembles["ensembles"][nom]
        lignes.append({
            "sous-jeu": sous_jeu, "ensemble": nom, "LSTM seul (g42)": round(seul, 3),
            "ensemble (neufs)": round(float(np.mean(resultat["rmse_cv_par_decoupage"])), 3),
            "gain": round(resultat["comparaison"]["gain_moyen"], 3),
            "p Wilcoxon": f"{resultat['comparaison']['p_wilcoxon']:.1e}",
            "RETENU contre un seul LSTM": resultat["comparaison"]["retenu"],
        })
tableau_ensembles = pd.DataFrame(lignes)
tableau_ensembles

**Lecture et choix.**

- **FD002** : seul l'ensemble de 5 LSTM passe la règle ; ajouter le CNN (non optimisé sur FD002) le dégrade. Retenu : **5 LSTM**.
- **FD001** : les deux ensembles passent la règle contre un seul LSTM. Entre eux, l'écart de validation est de 0,02 cycle (7,38 contre 7,35), très loin du seuil de 0,3 : rien ne justifie la version à 10 réseaux, plus lourde. On garde la plus simple : **5 LSTM**. Ce choix est fait ici, sur la validation, **avant** de regarder le test (section 7), où l'ensemble à 10 réseaux se trouve être un peu meilleur : le choisir pour cette raison reviendrait à choisir sur le test.

## 7. Le test officiel, lu une seule fois

Les modèles finaux sont entraînés sur **tous** les moteurs du train, pendant un nombre d'époques fixé par la validation (médiane des meilleures époques des 5 plis de la graine 42), puis notés au dernier cycle de chaque moteur test, comme le benchmark. Deux RMSE : contre la vraie RUL brute (le chiffre officiel), et contre la RUL plafonnée à 125 (ce que le modèle peut atteindre, puisque sa cible est plafonnée).

Les anciens résultats sont gardés à côté, pour la comparaison.

In [ ]:
# anciens resultats : notebook 03b (fichier 03b_deep_learning.csv) et XGBoost final (src/config_modele.py)
anciens = pd.read_csv(os.path.join(DOSSIER, "03b_deep_learning.csv"))
lignes = []
for _, ligne in anciens.iterrows():
    lignes.append({"sous-jeu": ligne["sous_jeu"], "modele": RESEAUX[ligne["modele"]] + " du 03b (avant)",
                   "RMSE brute": round(ligne["rmse_test"], 2), "RMSE plafonnee": round(ligne["rmse_test_rul_plafonnee"], 2),
                   "score NASA": round(ligne["nasa_test"], 1), "retenu": ""})
lignes.append({"sous-jeu": "FD001", "modele": "XGBoost final (04f)", "RMSE brute": CONFIG_XGBOOST_FINAL["metriques_test"]["rmse"],
               "RMSE plafonnee": None, "score NASA": CONFIG_XGBOOST_FINAL["metriques_test"]["score_nasa"], "retenu": ""})

# nouveaux resultats : chaque reseau seul (graine 42), la moyenne et l'ecart-type sur 5 graines, les ensembles
NOMS = {"lstm_g42": "LSTM Optuna, graine 42", "cnn1d_g42": "CNN 1D, graine 42",
        "lstm_5_graines": "ensemble 5 LSTM", "lstm_cnn_10": "ensemble 5 LSTM + 5 CNN"}
for sous_jeu in PREFIXES:
    test = lire_json(sous_jeu, "test")["metriques_test"]
    for cle in NOMS:
        nom = NOMS[cle]
        if sous_jeu == "FD002" and cle == "cnn1d_g42":
            nom = "CNN 1D (reglages du 03b), graine 42"  # non confirme sur FD002
        lignes.append({"sous-jeu": sous_jeu, "modele": nom, "RMSE brute": round(test[cle]["rmse"], 2),
                       "RMSE plafonnee": round(test[cle]["rmse_contre_rul_plafonnee"], 2),
                       "score NASA": round(test[cle]["score_nasa"], 1),
                       "retenu": "OUI" if cle == "lstm_5_graines" else ""})
    for reseau in RESEAUX:  # robustesse a la graine : 5 modeles de la meme configuration
        valeurs = []
        for graine in [42, 43, 44, 45, 46]:
            valeurs.append(test[f"{reseau}_g{graine}"]["rmse"])
        print(f"{sous_jeu}, {RESEAUX[reseau]} seul, 5 graines : RMSE brute {np.mean(valeurs):.2f} +- {np.std(valeurs):.2f} "
              f"(de {min(valeurs):.2f} a {max(valeurs):.2f})")
tableau_test = pd.DataFrame(lignes)
tableau_test

**Deux remarques sur les anciens chiffres.**

- Le CNN 1D du 03b vaut ici 13,09 au test FD001, alors que `STATUS.md` (entrée du 2026-11-05) indique 12,74. Le fichier `03b_deep_learning.csv` a été réécrit par un entraînement plus récent du 03b (même date, 3 graines, moyenne 12,18). On affiche le fichier, sans retoucher aucun des deux chiffres ; l'écart montre surtout combien le CNN du 03b dépend de sa graine.
- Le 03b n'a pas enregistré de test des réseaux sur FD002 : pour FD002, la comparaison avant / après se fait seulement en validation (section 4).

**Lecture.**

- **FD001** : l'ensemble de 5 LSTM fait **11,16** (10,09 plafonné), contre 12,29 pour le LSTM du 03b et 12,73 pour le XGBoost final. Le gain vient d'abord d'Optuna (12,29 → 11,46 pour un LSTM seul), puis de l'ensemble (11,46 → 11,16).
- Le CNN 1D seul est très sensible à la graine (12,89 ± 1,58 sur 5 graines) : c'est une raison de plus de ne pas l'utiliser seul.
- **FD002** : 22,38 brut mais 10,62 plafonné. L'écart vient des moteurs test dont la vraie RUL dépasse 125 : 57 sur 259 (22 %) sur FD002, contre 11 sur 100 sur FD001 (figure ci-dessous). Le modèle, entraîné sur une cible plafonnée, ne prédit jamais beaucoup plus de 125.

In [ ]:
# --- Figure : prediction au dernier cycle contre la vraie RUL, ensemble retenu ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, sous_jeu in enumerate(PREFIXES):
    predictions = pd.read_csv(os.path.join(DOSSIER, PREFIXES[sous_jeu] + "test_dernier_cycle.csv"))
    vraie = predictions["rul_vraie"].to_numpy()
    predite = predictions["lstm_5_graines"].to_numpy()
    au_dessus = vraie > opt.PLAFOND_RUL  # moteurs dont la vraie RUL depasse le plafond
    ax = axes[i]
    ax.scatter(vraie[~au_dessus], predite[~au_dessus], s=14, color=COULEURS["lstm"], label="RUL vraie <= 125")
    ax.scatter(vraie[au_dessus], predite[au_dessus], s=14, color="#D55E00", label="RUL vraie > 125")
    limite = max(vraie.max(), predite.max()) + 10
    ax.plot([0, limite], [0, limite], color="black", linewidth=1, label="prediction parfaite")
    ax.axvline(opt.PLAFOND_RUL, color=COULEURS["reference"], linestyle="--")
    ax.set_xlabel("RUL vraie (cycles)")
    ax.set_ylabel("RUL predite par l'ensemble de 5 LSTM")
    ax.set_title(f"{sous_jeu} : {int(au_dessus.sum())} moteurs test sur {len(vraie)} au-dessus de 125")
    ax.legend(fontsize=8)
fig.suptitle("Ou l'ensemble retenu se trompe-t-il au test ?")
plt.tight_layout()
plt.show()

**Lecture.** Sous 125 cycles, les points suivent la diagonale, et les erreurs grandissent avec la RUL (un moteur encore loin de la panne donne peu de signal). Au-dessus de 125, les prédictions restent bloquées vers 125 : c'est le plafond de la cible, choisi en Phase 2. FD002 a beaucoup plus de moteurs test dans ce cas que FD001, ce qui explique presque tout son RMSE brut élevé.

## 8. L'évolution, étape par étape (FD001)

Une seule chose change à chaque étape. On montre la validation **et** le test côte à côte : les choix ont été faits sur la validation, le test n'est qu'un constat. La validation est celle des 3 découpages neufs, la même pour les trois étapes (la référence du 03b y a été réévaluée à l'identique).

In [ ]:
confirmation = lire_json("FD001", "confirmation_lstm")
ensembles = lire_json("FD001", "ensemble")
test = lire_json("FD001", "test")["metriques_test"]
lstm_03b = anciens[(anciens["sous_jeu"] == "FD001") & (anciens["modele"] == "lstm")].iloc[0]

etapes = ["LSTM du 03b\n(reglages fixes)", "LSTM Optuna\n(1 graine)", "ensemble\n5 LSTM"]
validation = [moyenne_neufs(confirmation["cv_reference"]), moyenne_neufs(confirmation["cv_candidat"]),
              float(np.mean(ensembles["ensembles"]["lstm_5_graines"]["rmse_cv_par_decoupage"]))]
test_brut = [lstm_03b["rmse_test"], test["lstm_g42"]["rmse"], test["lstm_5_graines"]["rmse"]]

positions = np.arange(len(etapes))
fig, ax = plt.subplots(figsize=(9, 5))
barres_val = ax.bar(positions - 0.2, validation, width=0.4, color="#56B4E9", label="validation (3 decoupages neufs)")
barres_test = ax.bar(positions + 0.2, test_brut, width=0.4, color=COULEURS["lstm"], label="test officiel (RUL brute)")
ax.axhline(CONFIG_XGBOOST_FINAL["metriques_test"]["rmse"], color="#009E73", linestyle="--",
           label=f"XGBoost final, test ({CONFIG_XGBOOST_FINAL['metriques_test']['rmse']})")
for barres in [barres_val, barres_test]:
    for barre in barres:
        ax.annotate(f"{barre.get_height():.2f}", (barre.get_x() + barre.get_width() / 2, barre.get_height()),
                    ha="center", va="bottom", fontsize=9)
ax.set_xticks(positions)
ax.set_xticklabels(etapes)
ax.set_ylabel("RMSE (cycles)")
ax.set_ylim(0, 14)
ax.legend(loc="lower left", fontsize=9)
ax.set_title("FD001 : comment le RMSE a-t-il baisse, etape par etape ?")
plt.tight_layout()
plt.show()

**Lecture.**

- Les deux séries baissent ensemble, dans le même ordre : ce que la validation annonçait, le test le confirme.
- Mais **le gain au test est plus petit que le gain en validation** : 1,13 cycle au test (12,29 → 11,16) contre 1,96 en validation (9,34 → 7,38). L'écart entre validation et test passe ainsi de 3,0 cycles au 03b à 3,6 puis 3,8. Une part du gain de validation ne se retrouve donc pas au test : c'est attendu quand on choisit parmi beaucoup d'essais, et c'est la raison pour laquelle on annonce le chiffre du test, pas celui de la validation.
- Pourquoi la validation est plus basse que le test : elle porte sur toutes les lignes (beaucoup de cycles en début de vie, où la cible plafonnée vaut 125 et se prédit bien), le test seulement sur le dernier cycle de chaque moteur test, dont la vraie RUL n'est pas plafonnée.

Sur FD002, la même évolution en validation : LSTM du 03b 10,28 → LSTM Optuna 9,90 → ensemble de 5 LSTM 9,50 (section 6). Pas de test « avant » enregistré pour FD002.

## 9. Comparaison à la littérature (FD001)

Chiffres relevés au notebook 05, section H (références complètes là-bas) : DCNN (Li, Ding, Sun 2018) 12,61 ; DAST (Zhang, Song, Li 2022) 11,43 ; STAGNN (Huang, He, Sick 2023) 11,50.

| Modèle | RMSE test FD001 |
|---|---|
| DCNN (2018) | 12,61 |
| STAGNN (2023) | 11,50 |
| DAST (2022) | 11,43 |
| **Ensemble de 5 LSTM (ce notebook)**, contre la RUL brute | **11,16** |
| **Ensemble de 5 LSTM (ce notebook)**, contre la RUL plafonnée à 125 | **10,09** |

**Prudence.** Le notebook 05 note que ces articles plafonnent probablement aussi la vérité du test : le chiffre comparable serait alors 10,09. Dans les deux cas, on est au niveau ou en dessous de ces trois articles. Mais leurs protocoles ne sont pas tous identiques au nôtre (fenêtres, plafond, nombre de répétitions) : la comparaison est indicative, pas un classement.

## 10. Où sont les réglages retenus, et comment réentraîner sans Optuna

Les réglages retenus sont recopiés, sans arrondi et commentés, dans `src/optimisation_reseaux.py` : `CONFIGS_RESEAUX_RETENUES` (par sous-jeu et par réseau, avec le nombre d'époques du modèle final) et `ENSEMBLES_RETENUS`. La cellule suivante vérifie qu'ils sont identiques à ceux que le calcul de nuit a réellement utilisés.

Réentraîner un modèle final prend quelques minutes, sans relancer Optuna :

```python
from src import optimisation_reseaux as opt
config = opt.CONFIGS_RESEAUX_RETENUES["FD001"]["lstm"]
predictions = []
for graine in opt.ENSEMBLES_RETENUS["FD001"]["graines"]:
    modele = opt.entrainer_modele_final(config, config["nb_epoques_final"], graine=graine, sous_jeu="FD001")
    predictions.append(opt.predictions_test_dernier_cycle(modele, config, sous_jeu="FD001"))
moyenne = np.mean(np.vstack(predictions), axis=0)  # l'ensemble de 5 LSTM
```

Les poids des réseaux ne sont pas sauvegardés : la démo (`sorties/predictions/`, `models/`) n'utilise pas encore ces réglages.

In [ ]:
for sous_jeu in opt.CONFIGS_RESEAUX_RETENUES:
    utilisees = lire_json(sous_jeu, "test")["configurations"]  # configurations du calcul de nuit (etape final)
    for reseau in opt.CONFIGS_RESEAUX_RETENUES[sous_jeu]:
        figee = dict(opt.CONFIGS_RESEAUX_RETENUES[sous_jeu][reseau])
        nb_epoques = figee.pop("nb_epoques_final")
        identique = json.loads(json.dumps(figee)) == utilisees[reseau]
        print(f"{sous_jeu}, {RESEAUX[reseau]} : identique au calcul de nuit = {identique}, {nb_epoques} epoques, "
              f"{figee['hyperparametres']}, entrees {figee['entrees']}")
print("ensembles retenus :", opt.ENSEMBLES_RETENUS)

## 11. Limites

1. **FD003 et FD004 n'ont pas de recherche propre** : leurs réseaux gardent les réglages du 03b, choisis sur FD001. C'est une limite à dire au jury ; la méthode et le code (`--sous-jeu FD003`) sont prêts, il manque environ 5 h de calcul pour FD003 et 12 h pour FD004.
2. **Une seule recherche par étude.** Optuna lui-même a une part de hasard ; une autre graine de l'échantillonneur aurait pu trouver un autre optimum. La confirmation sur des découpages neufs protège contre un faux gain, pas contre un optimum manqué.
3. **Le test FD001 compte 100 moteurs** : un écart de quelques dixièmes au test (par exemple 11,16 contre 10,97 pour l'ensemble à 10 réseaux) n'est pas significatif. La décision repose sur la validation.
4. **Le RMSE brut de FD002 est dominé par le plafond de 125** (section 7) ; améliorer ce chiffre demanderait de changer la cible, pas les réglages des réseaux.
5. **La recherche du CNN de FD001 n'était peut-être pas finie** (dernier record à l'essai 77 sur 89, section 2) ; quelques dizaines d'essais de plus auraient pu l'améliorer encore.
6. **Le gain au test est plus petit que le gain en validation** (section 8) : il faut annoncer le chiffre du test.
7. **Architectures de la littérature** : le code contient un CNN suivi d'un LSTM et un LSTM avec attention (`src/optimisation_reseaux.py`), mais ils n'ont pas été recherchés par Optuna, faute de temps de calcul.
8. **Les poids ne sont pas sauvegardés** et la démo n'utilise pas encore ces réglages (section 10).